# Self-Play (Colab) — reusable Q-recording generator (V15, V16, V17, …)

Generates self-play that is **Gumbel-ready as written** — each move records `cand_prior`
(clean, **pre-Dirichlet**), `cand_q` (root Q), visits, root_value, q_min/q_max. **No relabel
step.** (Relabel only exists for *legacy* V13/V14 data that predates Q recording.)

Why this is the right recipe, kept fixed across generations:
- The **prior** is captured before Dirichlet → clean label regardless of exploration.
- **Q** comes from the search; Dirichlet broadens *which* moves get simulated (better Q
  coverage of low-prior improvement moves) without biasing the values. The Gumbel target uses
  **prior + Q**, not noisy visits, so exploration doesn't leak into the label.
- So Dirichlet + temperature stay **ON** — they diversify states and improve Q-coverage.

**To run a new generation: edit the 3 fields in the config cell** (VERSION / MODEL /
VALUE_HEAD) and bump SEED_START. Everything else is the fixed recipe.

**Upload to `MyDrive/alphatrain/`:** the current **Q-recording** `colorlines_pillar3d_v2.tar.gz`,
plus the generation's `MODEL` + `VALUE_HEAD` (the value head must be retrained on MODEL's
backbone each generation — HISTORY 158).

In [ ]:
# ==== EDIT PER GENERATION ====
VERSION      = 15                          # 15, 16, 17, …  -> save dir selfplay_v{VERSION}
MODEL        = 'pillar3f.pt'               # current best policy (V16: pillar3g.pt, …)
VALUE_HEAD   = 'value_head_pillar3f.pt'    # retrained on MODEL's backbone (HISTORY 158)
SEED_START   = 1700000                     # bump each generation so games stay disjoint
SEED_END     = 1700020                     # canary first (~20); widen to e.g. +200/+1000 after
# ---- fixed recipe (rarely changes) ----
SIMS=400; Q_WEIGHT=2.0; MAX_TURNS=10000; TEMP_MOVES=10; WORKERS=24; BATCH_SIZE=8

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
# GUARD: the tarball must be the Q-recording build, else self-play silently
# produces no-Q games (wasting the run). Fail loud if a stale tarball was uploaded.
assert 'last_root_record' in open('/content/alphatrain/scripts/selfplay.py').read(), \
    'STALE TARBALL: re-upload the Q-recording colorlines_pillar3d_v2.tar.gz.'
print('Q-recording recipe confirmed (selfplay records cand_prior/cand_q).')
os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/{MODEL} /content/alphatrain/data/{MODEL}
!cp {DRIVE}/{VALUE_HEAD} /content/alphatrain/data/{VALUE_HEAD}
print('model', MODEL, os.path.getsize(f'/content/alphatrain/data/{MODEL}')//1_000_000, 'MB')
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Generate (canary first; resume-safe on Drive)

In [ ]:
SAVE_DIR = f'{DRIVE}/selfplay_v{VERSION}'
# If a prior (aborted) run wrote games here, CLEAR them first — resume skips
# existing files, so stale/old-schema games would silently stay in the corpus.
import os; os.makedirs(SAVE_DIR, exist_ok=True)
print(f'V{VERSION}: seeds {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games), '
      f'model {MODEL}, sims {SIMS}, cap {MAX_TURNS} -> {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/{MODEL} \
    --value-head-path /content/alphatrain/data/{VALUE_HEAD} \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} --max-turns {MAX_TURNS} \
    --compile

## Notes
- **Canary:** run the ~20-seed shard first; per-game scores should land in/above the model's
  policy-only ballpark (capped games print `[CAPPED]`). If far below, stop and q-sweep.
- **Resume:** re-run the generate cell after a disconnect — it skips games already in `SAVE_DIR`
  (on Drive). Shard across instances by editing SEED_START/SEED_END.
- **GPU health:** watch `[GPU] … avg bs=Z, … evals/s`; avg bs ≥50 means the GPU is fed.

## After generation
Build the SLIM Q tensor (drops ~4GB value/pairwise; keeps `cand_prior`/`cand_q`):
```
python -m alphatrain.scripts.build_expert_v2_tensor \
    --games-dir data/selfplay_v{VERSION} [data/crisis_v{VERSION}] [data/relabel_v15] \
    --policy-only-data --output alphatrain/data/v{VERSION}_{MODEL%.pt}.pt
```
(`relabel_v15` = the one-time clean-Q re-search of legacy V14, optional extra coverage.) Then the
**Gumbel/advantage trainer**: target softmax(prior + Q), disagreement-weight, KL-anchor, no
sharpening, warm-start the trunk from the trained base. Build with `--policy-only-data`'s
all-zero-Q warning as the guard against any stray no-Q games.